## Funções auxiliares

In [5]:
def step(x):
  return 1 if x > 0 else 0

def derivada_step(x):
  return 0

def relu(x):
  return x if x > 0 else 0

def derivada_relu(x):
  return 1 if x > 0 else 0

activation_functions = {
    "step": (step, derivada_step),
    "relu": (relu, derivada_relu)
}

## MLP

In [4]:
import numpy as np

class MLP:
    def __init__(self, layers, activations, activation_functions):
        self.layers = layers
        self.weights = []
        self.biases = []
        self.activation_forward = []
        self.activation_backward = []

        for rows, cols in zip(self.layers[:-1], self.layers[1:]):
            self.weights.append(np.random.randn(rows, cols))
            self.biases.append(np.zeros(cols))

        for name in activations:
            func_forward, func_backward = activation_functions[name]
            self.activation_forward.append(func_forward)
            self.activation_backward.append(func_backward)

    def forward(self, x):
        h_a = [x]
        h_z = []
        for w, b, f in zip(self.weights, self.biases, self.activation_forward):
            z = x @ w + b
            h_z.append(z)
            x = f(z)
            h_a.append(x)
        return x, h_z, h_a

    def backward(self, h_a, h_z, y_true):
        y_pred = h_a[-1]
        error = y_pred - y_true
        delta = error * self.activation_backward[-1](h_z[-1])

        gradientes_w = []
        gradientes_b = []

        for i in reversed(range(len(self.weights))):
            ativacao_entrada = h_a[i]

            gradiente_w = np.atleast_2d(ativacao_entrada).T @ np.atleast_2d(delta)
            gradientes_w.insert(0, gradiente_w)

            gradiente_b = np.sum(delta, axis=0, keepdims=True)
            gradientes_b.insert(0, gradiente_b)

            if i > 0:
                delta = (delta @ self.weights[i].T) * self.activation_backward[i-1](h_z[i-1])

        return gradientes_w, gradientes_b

    def update_weights(self, gradientes_w, gradientes_b, learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * gradientes_w[i]
            self.biases[i] -= learning_rate * gradientes_b[i]

    def fit(self, X, y, epochs=1000, learning_rate=0.1):
        for epoch in range(epochs):
            erro_epoca = 0

            for x_i, y_i in zip(X, y):
                y_pred, h_z, h_a = self.forward(x_i)

                grad_w, grad_b = self.backward(h_a, h_z, y_i)

                self.update_weights(grad_w, grad_b, learning_rate)

                erro_epoca += np.mean((y_pred - y_i) ** 2)

            if (epoch + 1) % 100 == 0:
                print(f"Época {epoch + 1}/{epochs} - Erro: {erro_epoca / len(X):.6f}")

    def predict(self, X):
        previsoes = []
        for x_i in X:
            y_pred, _, _ = self.forward(x_i)
            previsoes.append(y_pred)
        return np.array(previsoes)

## Comparando

In [10]:
import numpy as np
from sklearn.neural_network import MLPClassifier

def sigmoid(x):
    x_clipped = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x_clipped))

def derivada_sigmoid(x):
    sig = sigmoid(x)
    return sig * (1 - sig)

activation_functions = {
    "sigmoid": (sigmoid, derivada_sigmoid)
}

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([[0],
              [1],
              [1],
              [0]])

print("MLP:")

meu_mlp = MLP(
    layers=[2, 4, 1],
    activations=['sigmoid', 'sigmoid'],
    activation_functions=activation_functions
)

meu_mlp.fit(X, y, epochs=5000, learning_rate=0.5)

minhas_previsoes_brutas = meu_mlp.predict(X)

minhas_previsoes_finais = np.round(minhas_previsoes_brutas)

print("\nPredições:")
for i in range(len(X)):
    print(f"Entrada: {X[i]} | Bruto: {minhas_previsoes_brutas[i][0]:.4f} -> Arredondado: {minhas_previsoes_finais[i][0]}")


print("\nSCIKIT-LEARN:")

sk_mlp = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation='logistic',
    learning_rate_init=0.5,
    max_iter=5000,
    random_state=42
)

sk_mlp.fit(X, y.ravel())

sk_previsoes = sk_mlp.predict(X)

print("\nPredições:")
for i in range(len(X)):
    print(f"Entrada: {X[i]} | Previsão: {sk_previsoes[i]}")

MLP:
Época 100/5000 - Erro: 0.252140
Época 200/5000 - Erro: 0.223746
Época 300/5000 - Erro: 0.153183
Época 400/5000 - Erro: 0.060246
Época 500/5000 - Erro: 0.026862
Época 600/5000 - Erro: 0.015535
Época 700/5000 - Erro: 0.010484
Época 800/5000 - Erro: 0.007755
Época 900/5000 - Erro: 0.006085
Época 1000/5000 - Erro: 0.004972
Época 1100/5000 - Erro: 0.004184
Época 1200/5000 - Erro: 0.003599
Época 1300/5000 - Erro: 0.003151
Época 1400/5000 - Erro: 0.002797
Época 1500/5000 - Erro: 0.002510
Época 1600/5000 - Erro: 0.002275
Época 1700/5000 - Erro: 0.002078
Época 1800/5000 - Erro: 0.001911
Época 1900/5000 - Erro: 0.001768
Época 2000/5000 - Erro: 0.001643
Época 2100/5000 - Erro: 0.001535
Época 2200/5000 - Erro: 0.001439
Época 2300/5000 - Erro: 0.001354
Época 2400/5000 - Erro: 0.001278
Época 2500/5000 - Erro: 0.001210
Época 2600/5000 - Erro: 0.001149
Época 2700/5000 - Erro: 0.001093
Época 2800/5000 - Erro: 0.001042
Época 2900/5000 - Erro: 0.000996
Época 3000/5000 - Erro: 0.000953
Época 3100/500